In [1]:
import pandas as pd
from h3 import h3
import glob
import numpy as np
from pandas import merge
import json
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import shape
from datetime import datetime, timedelta
import seaborn as sns

In [11]:
# Pandas Config
pd.set_option('display.max_columns', None)
pd.options.display.max_rows = 200

# ShareNow Data (results from Prepare_Trip_Data)
VEHICLE_TRIP = '/Users/demircanm/sciebo/IS3/Research_Projects/HeterogeneousFleetManagement/01_Fleet_Sizing/0_Data/Berlin/shared_vehicle_trips_berlin.csv'

# TIMEZONE
TIMEZONE = 'Europe/Berlin'


In [12]:
# Read Trips
trips = pd.read_csv(VEHICLE_TRIP)

/Users/demircanm/opt/anaconda3/envs/SharedMobility/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3146: DtypeWarning: Columns (4,21,22) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [66]:
# Clean Vehicle Type
trips["vehicleType"] = trips["vehicleType"].replace('bycicle', 'bicycle')

In [68]:
hex_start_column, hex_end_column = "h3_start_7", "h3_end_7"
day_of_week_column = "is_holiday"
date_column = "date_start"
vehicle_type = "vehicleType"



In [69]:
# Count outflow trips in spatio temporal resolution
outflow = trips.groupby([date_column, vehicle_type, day_of_week_column, hex_start_column]).agg({"id": "count"}).reset_index()

outflow.rename(columns={"id":"count"}, inplace=True)

# Reduce to WorkingDay/Non-Working-Day
outflow = outflow.groupby([day_of_week_column, vehicle_type, hex_start_column]).agg({"count":"median"})

In [70]:
# Count outflow trips in spatio temporal resolution
inflow = trips.groupby([date_column, vehicle_type, day_of_week_column, hex_end_column]).agg({"id": "count"}).reset_index()

inflow.rename(columns={"id":"count"}, inplace=True)

# Reduce to WorkingDay/Non-Working-Day
inflow = inflow.groupby([day_of_week_column, vehicle_type, hex_end_column]).agg({"count":"median"})

In [73]:
# Consider only hexagons that have data for each temporal data bucket
hexagon_to_consider = trips.groupby([hex_start_column]).agg({"id": "count"}).reset_index()
hexagon_to_consider["key"] = 1 # For Cross Join Later

temporal_dimension_to_consider = trips.groupby([day_of_week_column]).agg({"id": "count"}).reset_index()
temporal_dimension_to_consider["key"] = 1 # For Cross Join Later

vehicle_type_to_consider = trips.groupby([vehicle_type]).agg({"id": "count"}).reset_index()
vehicle_type_to_consider["key"] = 1 # For Cross Join Later

result = hexagon_to_consider.merge(temporal_dimension_to_consider, on="key", how="inner")
result = result.merge(vehicle_type_to_consider, on="key", how="inner")

result = result.drop(columns={"id_x", "key", "id_y", "id"})

In [74]:
result

,h3_start_7,is_holiday,vehicleType
0,871f18840ffffff,False,bicycle
1,871f18840ffffff,False,car
2,871f18840ffffff,False,kick scooter
3,871f18840ffffff,False,scooter
4,871f18840ffffff,True,bicycle
...,...,...,...
1059,871f1d591ffffff,False,scooter
1060,871f1d591ffffff,True,bicycle
1061,871f1d591ffffff,True,car
1062,871f1d591ffffff,True,kick scooter


In [75]:
# Merge Inflow
result = result.merge(inflow, left_on=[hex_start_column, vehicle_type, day_of_week_column], right_on=[hex_end_column, vehicle_type, day_of_week_column], how = "left")

result = result.rename(columns={"count":"inflow"})

# Merge Outflow
result = result.merge(outflow, left_on=[hex_start_column, vehicle_type, day_of_week_column], right_on=[hex_start_column, vehicle_type, day_of_week_column], how = "left")

result = result.rename(columns={"count":"outflow"})

# Fill NA
result = result.fillna(0)

In [76]:
result["diff"] = result["inflow"] - result["outflow"]

In [81]:
# Not used in Simulation so far...
result.to_csv('inflow_outflow_hex_7_daily.csv', index = False)